# 05 Baseline results report

Ticket T4.2.3. **Simulated, benchmark-parameterized line; not plant data.** Frozen baseline v1.0 (tag `baseline-v1.0`, config hash `2f9b035c9964`), 30 replications, seed 20261116 (design from T4.1.3). Every number is a RESULT read from `results/baseline/*.parquet` (written by `python tasks.py scenarios`) and exported to `reports/results_baseline.json`; `reports/baseline.md` is generated from that JSON. Confidence intervals are t-based 95% over replications.

In [1]:
from pathlib import Path

import pandas as pd
import plotly.graph_objects as go

from ffdt import viz
from ffdt.experiments import baseline_report as B

RES, REP = Path("../results"), Path("../reports")
t = B.load(RES)
data = B.write(RES, REP / "results_baseline.json")
(REP / "baseline.md").write_text(B.markdown(data), encoding="utf-8")
pd.set_option("display.width", 200)
print(data["label"], "| n_reps:", data["n_reps"], "| all completed:", data["all_completed"])
B.line_table(t["summary"]).round(4)

Simulated, benchmark-parameterized. Not plant data. | n_reps: 30 | all completed: True


,mean,ci_low,ci_high
line_oee,0.5731,0.5643,0.5818
line_A,0.7127,0.7031,0.7224
line_P,0.8068,0.7931,0.8206
line_Q,0.9974,0.9974,0.9974
good_per_ppt_hour,8252.2042,8126.6123,8377.7960
good_per_calendar_hour,3804.9788,3529.7278,4080.2298
makespan_h,161.6685,152.9027,170.4344
yield,0.9974,0.9974,0.9974
mean_cycle_time_h,31.8555,29.9926,33.7183
mean_lead_time_h,89.1697,78.5692,99.7702


## OEE A/P/Q per station (mean and 95% CI)

In [2]:
so = B.station_oee_table(t["stations"])
display(so.round(4))
s = so.drop(index="manual_inspection")
fig = go.Figure(
    go.Bar(
        x=s.index,
        y=100 * s.OEE,
        error_y=dict(
            type="data",
            symmetric=False,
            array=100 * (s.OEE_ci_high - s.OEE),
            arrayminus=100 * (s.OEE - s.OEE_ci_low),
            color="#52514e",
            thickness=1.5,
            width=6,
        ),
        marker=dict(color=viz.series_color(), cornerradius=4),
        text=[f"{v:.0f}%" for v in 100 * s.OEE],
        textposition="outside",
        cliponaxis=False,
        customdata=s[["A", "P", "Q"]].values * 100,
        hovertemplate="%{x}<br>OEE %{y:.1f}%<br>A %{customdata[0]:.1f}% · P %{customdata[1]:.1f}% · Q %{customdata[2]:.2f}%<extra></extra>",
    )
)
viz._layout(fig, "Station OEE, baseline v1.0 (95% CI over 30 replications)", 380)
fig.update_yaxes(title="OEE", ticksuffix="%", range=[0, 75], showgrid=True)
fig.update_xaxes(showgrid=False)
fig.update_layout(template="plotly_white", plot_bgcolor="#fcfcfb", paper_bgcolor="#fcfcfb")
fig.write_image(REP / "figures/05_station_oee.png", scale=2, width=820)
fig

,A,A_ci_low,A_ci_high,P,P_ci_low,P_ci_high,Q,Q_ci_low,Q_ci_high,OEE,OEE_ci_low,OEE_ci_high
station,,,,,,,,,,,,
washer,0.9055,0.8992,0.9119,0.4352,0.4285,0.4418,1.0000,1.0000,1.0000,0.3940,0.3880,0.4000
tunnel,0.9148,0.9087,0.9208,0.4308,0.4243,0.4373,1.0000,1.0000,1.0000,0.3940,0.3880,0.4000
filler,0.7127,0.7031,0.7224,0.8068,0.7931,0.8206,0.9990,0.9990,0.9990,0.5740,0.5653,0.5827
capper,0.8457,0.8389,0.8525,0.5432,0.5342,0.5522,1.0000,1.0000,1.0000,0.4592,0.4522,0.4662
avi,0.7979,0.7909,0.8050,0.5759,0.5646,0.5873,0.9883,0.9883,0.9884,0.4538,0.4469,0.4607
labeler,0.7873,0.7802,0.7944,0.5827,0.5719,0.5935,1.0000,1.0000,1.0000,0.4585,0.4515,0.4654
cartoner,0.7166,0.7094,0.7238,0.6861,0.6723,0.7000,1.0000,1.0000,1.0000,0.4912,0.4837,0.4987
manual_inspection,1.0000,1.0000,1.0000,0.1608,0.1582,0.1634,0.8628,0.8614,0.8643,0.1388,0.1365,0.1410


## Loss Pareto (six big losses, at the filler)

Shares of scheduled production time; they sum to 1 − line OEE. Starved and blocked are flow losses caused by other stations (glossary D3).

In [3]:
pt = B.pareto_table(t["pareto"])
display(pt.round(4))
fig = viz.pareto_fig(pt.rename(columns={"mean": "share_of_ppt"}))
fig.update_layout(template="plotly_white", plot_bgcolor="#fcfcfb", paper_bgcolor="#fcfcfb")
fig.write_image(REP / "figures/05_loss_pareto.png", scale=2, width=820)
fig

,category,six_big_loss,factor,mean,ci_low,ci_high
0,CHANGEOVER,Setup and adjustment,Availability,0.2737,0.2634,0.2840
1,BLOCKED,"Idling and minor stoppages (blocked, D3)",Performance,0.1001,0.0919,0.1082
2,MINOR_STOP,Idling and minor stoppages,Performance,0.0197,0.0183,0.0210
3,STARVED,"Idling and minor stoppages (starved, D3)",Performance,0.0185,0.0132,0.0237
4,DOWN,Equipment failure,Availability,0.0113,0.0049,0.0177
5,WAIT_STAFF,"Setup and adjustment (staff resource, D2)",Availability,0.0022,-0.0010,0.0054
6,avi_confirmed,Process defects,Quality,0.0009,0.0009,0.0009
7,fill_weight,Process defects,Quality,0.0006,0.0006,0.0006


## State time per station (share of scheduled production time)

In [4]:
sh = B.state_shares(t["stations"]).drop(index="manual_inspection")
display((100 * sh).round(1))
fig = go.Figure(
    [
        go.Bar(
            y=sh.index,
            x=100 * sh[s_],
            name=viz.STATE_LABELS[s_],
            orientation="h",
            marker=dict(color=viz.STATE_COLORS[s_], line=dict(color="#fcfcfb", width=1)),
            hovertemplate="%{y}: %{x:.1f}% " + viz.STATE_LABELS[s_] + "<extra></extra>",
        )
        for s_ in B.STATES
    ]
)
viz._layout(fig, "Where each station's scheduled time goes, baseline v1.0", 460)
fig.update_layout(
    barmode="stack",
    template="plotly_white",
    plot_bgcolor="#fcfcfb",
    paper_bgcolor="#fcfcfb",
    legend=dict(orientation="h", y=-0.2, x=0, traceorder="normal"),
)
fig.update_yaxes(autorange="reversed")
fig.update_xaxes(title="% of scheduled production time", ticksuffix="%", range=[0, 100])
fig.write_image(REP / "figures/05_station_states.png", scale=2, width=820)
fig

,RUN,STARVED,BLOCKED,DOWN,MINOR_STOP,CHANGEOVER,WAIT_STAFF
station,,,,,,,
washer,39.4,1.6,48.3,0.7,1.3,8.7,0.0
tunnel,39.4,3.3,48.8,0.4,0.0,8.1,0.0
filler,57.5,1.8,10.0,1.1,2.0,27.4,0.2
capper,45.9,26.9,10.2,0.5,1.6,14.9,0.0
avi,45.9,25.0,7.3,0.6,1.6,15.5,4.2
labeler,45.8,25.8,5.5,0.3,1.6,14.8,6.1
cartoner,49.1,21.0,0.0,0.7,1.6,15.1,12.6


## Preliminary bottleneck evidence (no conclusion here)

Three classic signals side by side: utilization (share of scheduled time running), starvation (waiting for input) and blocking (output buffer full), plus the time-averaged queue in front of each station. The constraint is identified in T5.2.3 by at least three methods; with random failures it can move. The queue in front of the washer is vials waiting to be released into the line, near its cap by construction, so it is not evidence.

In [5]:
B.bottleneck_evidence(t["stations"], t["queues"]).round(3)

,rate_vpm,utilization,starved_share,blocked_share,mean_queue_in_front_sublots
station,,,,,
washer,350.0,0.394,0.016,0.483,18.701
tunnel,350.0,0.394,0.033,0.488,16.752
filler,240.0,0.575,0.018,0.100,18.705
capper,300.0,0.459,0.269,0.102,2.855
avi,300.0,0.459,0.250,0.073,4.273
labeler,300.0,0.458,0.258,0.055,3.984
cartoner,280.0,0.491,0.210,0.000,4.442
